# CAT-2 — Polaris + RBAC (control-plane / admin-API playground)

> **This is a control-plane playground.** Iceberg's REST spec has **no `GRANT` SQL** — grants aren't a query-engine surface, they're an admin-API surface. So every RBAC operation in this notebook is an **HTTP call to Polaris's management API** (`urllib`, `application/json`, `Authorization: Bearer <token>`). Data queries against Polaris still use **Spark SQL** — we run those as `root` (the pre-configured admin credential) to prove Polaris is a real Iceberg catalog behind the RBAC layer.

Polaris's headline feature vs plain Iceberg REST is **fine-grained RBAC**: principals (users), catalog-roles (privileges scoped to a catalog), principal-roles (the binding between them), and grants themselves. This module builds a least-privileged reader principal, watches it 403 on a mutation, reads the exact denial message Polaris emits, grants the missing privilege, and proves the same call now succeeds.

**Break → Detect → Fix → Prove:**
1. **Break** — alice (reader only) → `DROP NAMESPACE cat2_demo` → **403 Forbidden**.
2. **Detect** — parse the 403 body; Polaris names the principal, the activated grants, and the denied operation (`DROP_NAMESPACE`).
3. **Fix** — grant `NAMESPACE_DROP` to alice's catalog-role (least-privilege, not admin escalation).
4. **Prove** — same DROP now returns 204; her `NAMESPACE_LIST` read still returns 200.

**Prereqs:** `make up && make catalogs-up`. Polaris healthy on `:8181`, root principal bootstrapped.

**Cross-cutting note.** The Polaris **data-plane** write path is STS-blocked in MiniStack Community (see `docs/PLAN_V2.md` §WS1) — we can't create a *table* through Polaris, only namespaces. **The RBAC semantics are identical for `TABLE_DROP` vs `NAMESPACE_DROP` — same wire call shape, same grants-check code path — so we exercise the pattern on a namespace and label it honestly.** For governed marts, this repo now uses `glue_catalog` (see [CAT-5](cat5_federation.ipynb)).

In [1]:
import json
import time
import urllib.error
import urllib.parse
import urllib.request
from common.spark_session import spark   # only used at the very end, for the SHOW NAMESPACES read via a real engine

POLARIS = "http://localhost:8181"
REALM = "default"
CATALOG = "lab"

# Root principal — bootstrapped by the catalogs profile.
ROOT_CLIENT_ID, ROOT_CLIENT_SECRET = "root", "secret"

# Names we'll create in this notebook (idempotent — deleted at start).
PRINCIPAL = "alice"                # our least-privileged persona
PRINCIPAL_ROLE = "reader_role"     # the role bound to alice (a job title)
CATALOG_ROLE = "reader"            # scoped role holding privileges on `lab`
DEMO_NS = "cat2_demo"              # the namespace we'll try to DROP


def request(method, path, token=None, body=None, form=None):
    """Thin urllib wrapper. Returns (status, decoded-json-or-empty). Never raises
    on HTTP errors — inspecting 403 bodies is the whole point of this lesson."""
    url = f"{POLARIS}{path}"
    headers = {"Polaris-Realm": REALM, "Accept": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    data = None
    if form is not None:
        headers["Content-Type"] = "application/x-www-form-urlencoded"
        data = urllib.parse.urlencode(form).encode()
    elif body is not None:
        headers["Content-Type"] = "application/json"
        data = json.dumps(body).encode()
    req = urllib.request.Request(url, data=data, method=method, headers=headers)
    try:
        with urllib.request.urlopen(req) as r:
            raw, status = r.read(), r.status
    except urllib.error.HTTPError as e:
        raw, status = e.read(), e.code
    try:
        return status, (json.loads(raw) if raw else {})
    except json.JSONDecodeError:
        return status, {"_raw": raw.decode(errors="replace")}


def get_token(client_id, client_secret):
    status, body = request("POST", "/api/catalog/v1/oauth/tokens", form={
        "grant_type": "client_credentials",
        "client_id": client_id, "client_secret": client_secret,
        "scope": "PRINCIPAL_ROLE:ALL",
    })
    assert status == 200 and "access_token" in body, f"OAuth2 failed for {client_id!r}: {status} {body}"
    return body["access_token"]


# Idempotent reset — delete leftovers from a prior run (silently, 404s are fine).
def reset():
    admin = get_token(ROOT_CLIENT_ID, ROOT_CLIENT_SECRET)
    for path in [
        f"/api/management/v1/principals/{PRINCIPAL}",
        f"/api/management/v1/principal-roles/{PRINCIPAL_ROLE}",
        f"/api/management/v1/catalogs/{CATALOG}/catalog-roles/{CATALOG_ROLE}",
    ]:
        request("DELETE", path, token=admin)
    request("DELETE", f"/api/catalog/v1/{CATALOG}/namespaces/{DEMO_NS}", token=admin)
    return admin


admin_token = reset()
print(f"admin token acquired ({len(admin_token)} chars); prior state cleaned.")

admin token acquired (654 chars); prior state cleaned.


## 1. Data-plane sanity — Spark can read the catalog

Before diving into RBAC, prove Polaris is a real Iceberg REST catalog: run a **Spark SQL** read against it. `spark.sql("SHOW NAMESPACES IN polaris_catalog")` goes through Polaris's data-plane catalog API — the same one the RBAC layer guards. This is what a real query engine looks like against Polaris; every RBAC check below is deciding whether calls of this shape are allowed.

> The Spark session in `spark-defaults.conf` is configured with `root:secret` credentials, so this succeeds — root has admin privileges on `lab`. When we set up alice below, her REST calls go through the *same code path*; RBAC is the only thing that changes the outcome.

In [2]:
# Data-plane read as root, via a real engine.
print("SHOW NAMESPACES IN polaris_catalog  (Spark SQL → Polaris data-plane, as root):")
for row in spark.sql("SHOW NAMESPACES IN polaris_catalog").collect():
    print("   ", row.asDict())

SHOW NAMESPACES IN polaris_catalog  (Spark SQL → Polaris data-plane, as root):


## 2. The RBAC model in one paragraph

```
 Principal (alice) ── attach ──▶ Principal-role (reader_role)
 │
 │ bind (per catalog)
 ▼
 Catalog-role (reader) on catalog `lab`
 │
 │ privileges
 ▼
 [TABLE_LIST, TABLE_READ_DATA, NAMESPACE_LIST, ...]
```

The two-level indirection (principal → **principal-role** → **catalog-role**) lets one `reader_role` job-title be bound to N catalogs' local `reader` catalog-roles without duplicating grants. Snowflake has the same pattern. Now we'll build alice and see the model fail closed.

In [3]:
# Seed the namespace alice will try to DROP.
status, body = request("POST", f"/api/catalog/v1/{CATALOG}/namespaces",
                       token=admin_token,
                       body={"namespace": [DEMO_NS], "properties": {}})
assert status in (200, 201), f"create namespace failed: {status} {body}"
print(f"[seed] namespace {CATALOG}.{DEMO_NS} created")

# Bind root → catalog_admin on this catalog. polaris-bootstrap creates the
# service_admin principal-role and puts root in it, but does NOT bind it to
# any catalog-role — without this, root can create principals but not create
# namespaces (409/204 both fine — idempotent).
request("PUT",
        f"/api/management/v1/principal-roles/service_admin/catalog-roles/{CATALOG}",
        token=admin_token,
        body={"catalogRole": {"name": "catalog_admin"}})
print("[seed] root → service_admin → catalog_admin bind ensured")

[seed] namespace lab.cat2_demo created
[seed] root → service_admin → catalog_admin bind ensured


## 3. Break — build alice as the least-privileged principal

Give alice **read** privileges only: `TABLE_LIST`, `TABLE_READ_DATA`, `TABLE_READ_PROPERTIES`, `NAMESPACE_LIST`, `NAMESPACE_READ_PROPERTIES`. Deliberately no `NAMESPACE_DROP` — that's the privilege whose absence we'll observe. Then have alice try to DROP the namespace and expect Polaris to refuse.

In [4]:
# 3a. Catalog-role holding reader privileges.
status, _ = request("POST",
                    f"/api/management/v1/catalogs/{CATALOG}/catalog-roles",
                    token=admin_token,
                    body={"catalogRole": {"name": CATALOG_ROLE}})
assert status in (201, 409)
for priv in ["TABLE_LIST", "TABLE_READ_DATA", "TABLE_READ_PROPERTIES",
             "NAMESPACE_LIST", "NAMESPACE_READ_PROPERTIES"]:
    status, _ = request("PUT",
                        f"/api/management/v1/catalogs/{CATALOG}/catalog-roles/{CATALOG_ROLE}/grants",
                        token=admin_token,
                        body={"grant": {"type": "catalog", "privilege": priv}})
    assert status in (200, 201, 204), f"grant {priv} failed: {status}"

_, grants = request("GET",
                    f"/api/management/v1/catalogs/{CATALOG}/catalog-roles/{CATALOG_ROLE}/grants",
                    token=admin_token)
print(f"catalog-role '{CATALOG_ROLE}' privileges:")
for g in grants.get("grants", []):
    print(f"   - {g['privilege']}")

# 3b. Alice's principal + principal-role, and bind them together.
status, body = request("POST", "/api/management/v1/principals",
                       token=admin_token,
                       body={"principal": {"name": PRINCIPAL}})
assert status == 201, f"create principal failed: {status} {body}"
creds = body["credentials"]   # {clientId, clientSecret} — only returned at create time!
print(f"\ncreated principal '{PRINCIPAL}' with clientId={creds['clientId']}")

for path, payload in [
    ("/api/management/v1/principal-roles", {"principalRole": {"name": PRINCIPAL_ROLE}}),
]:
    status, _ = request("POST", path, token=admin_token, body=payload)
    assert status in (201, 409)

request("PUT", f"/api/management/v1/principals/{PRINCIPAL}/principal-roles",
        token=admin_token, body={"principalRole": {"name": PRINCIPAL_ROLE}})
request("PUT",
        f"/api/management/v1/principal-roles/{PRINCIPAL_ROLE}/catalog-roles/{CATALOG}",
        token=admin_token, body={"catalogRole": {"name": CATALOG_ROLE}})
print(f"bound {PRINCIPAL} → {PRINCIPAL_ROLE} → {CATALOG_ROLE} on {CATALOG}")

catalog-role 'reader' privileges:
   - TABLE_LIST
   - TABLE_READ_DATA
   - TABLE_READ_PROPERTIES
   - NAMESPACE_LIST
   - NAMESPACE_READ_PROPERTIES

created principal 'alice' with clientId=4dc874ee72447a73
bound alice → reader_role → reader on lab


## 4. Alice tries to DROP — 403 Forbidden

Alice authenticates with **her own** `client_id` / `client_secret` (Polaris only reveals the secret at creation — real deployments persist it in a secret store immediately). Then she calls `DELETE /api/catalog/v1/lab/namespaces/cat2_demo`.

In [5]:
alice_token = get_token(creds["clientId"], creds["clientSecret"])
print(f"alice authenticated with her own credentials ({len(alice_token)} chars)\n")

status, body = request("DELETE",
                       f"/api/catalog/v1/{CATALOG}/namespaces/{DEMO_NS}",
                       token=alice_token)
print(f"alice → DELETE /namespaces/{DEMO_NS}  →  HTTP {status}")
print("response body:")
print(json.dumps(body, indent=2))

assert status == 403, f"expected 403, got {status}"
assert body["error"]["type"] == "ForbiddenException"

alice authenticated with her own credentials (671 chars)

alice → DELETE /namespaces/cat2_demo  →  HTTP 403
response body:
{
  "error": {
    "message": "Principal 'alice' with activated PrincipalRoles '[reader_role]' and activated grants via '[reader_role, reader]' is not authorized for op DROP_NAMESPACE",
    "type": "ForbiddenException",
    "code": 403
  }
}


## 5. Detect — the 403 body is the design

In a real audit-log incident ("why can't the marts job write?") the response body names:

- **the principal** (who tried)
- **the activated principal-roles** (which of the caller's roles Polaris walked)
- **the activated grants path** (`[reader_role, reader]` — the two-level binding)
- **the denied operation** (`DROP_NAMESPACE` — the exact enum name in Polaris's privilege taxonomy)

The enum verbatim is what you grep for. `DROP TABLE` → `TABLE_DROP`. `SELECT` → `TABLE_READ_DATA`. Same shape.

In [6]:
err = body["error"]
print(f"error.type    = {err['type']}")
print(f"error.code    = {err['code']}")
print(f"error.message = {err['message']}")
print()
print("    ↑ this string is the audit trail. It names the principal, the")
print("      activated roles, the walked grants path, and the denied op.")

error.type    = ForbiddenException
error.code    = 403
error.message = Principal 'alice' with activated PrincipalRoles '[reader_role]' and activated grants via '[reader_role, reader]' is not authorized for op DROP_NAMESPACE

    ↑ this string is the audit trail. It names the principal, the
      activated roles, the walked grants path, and the denied op.


## 6. Fix — the least-privilege grant

**Bad fix:** bind alice's role to `catalog_admin` — she'd gain `TABLE_WRITE_DATA` on every table too. Post-mortem material.

**Good fix:** grant *only* `NAMESPACE_DROP` to the reader catalog-role. Alice keeps her reads, gains exactly the one operation she needed. Additive, obvious in the audit log, easy to reverse.

In [7]:
status, _ = request("PUT",
                    f"/api/management/v1/catalogs/{CATALOG}/catalog-roles/{CATALOG_ROLE}/grants",
                    token=admin_token,
                    body={"grant": {"type": "catalog", "privilege": "NAMESPACE_DROP"}})
assert status in (200, 201, 204)

_, grants = request("GET",
                    f"/api/management/v1/catalogs/{CATALOG}/catalog-roles/{CATALOG_ROLE}/grants",
                    token=admin_token)
print(f"catalog-role '{CATALOG_ROLE}' privileges after grant:")
for g in grants.get("grants", []):
    print(f"   - {g['privilege']}")

# New grants are cached briefly on Polaris's side; give it a beat.
time.sleep(1)

catalog-role 'reader' privileges after grant:
   - TABLE_LIST
   - TABLE_READ_DATA
   - TABLE_READ_PROPERTIES
   - NAMESPACE_LIST
   - NAMESPACE_READ_PROPERTIES
   - NAMESPACE_DROP


## 7. Prove — same DROP now succeeds; alice's reads still work

Repeat the exact same DELETE. Same principal, same token, same URL — the only difference is the grant we added a moment ago. Then confirm her read privileges are untouched: her `NAMESPACE_LIST` call still returns 200 (the `cat2_demo` namespace is now gone because she just dropped it).

In [8]:
# 7a. Retry the DROP.
status, body = request("DELETE",
                       f"/api/catalog/v1/{CATALOG}/namespaces/{DEMO_NS}",
                       token=alice_token)
print(f"alice → DELETE /namespaces/{DEMO_NS}  →  HTTP {status}")
assert status in (200, 204), f"expected 200/204 after grant, got {status}: {body}"

# 7b. Read privilege still works — a real LIST call, not a mislabeled "SELECT-shaped" one.
status, body = request("GET", f"/api/catalog/v1/{CATALOG}/namespaces", token=alice_token)
print(f"alice → GET    /namespaces           →  HTTP {status}")
print(f"   alice sees namespaces: {body.get('namespaces', [])}")
assert status == 200, f"reader lost NAMESPACE_LIST somehow: {status} {body}"

print("\nSummary (BEFORE → AFTER):")
print(f"   DROP  as {PRINCIPAL} : 403 ForbiddenException  →  {status if False else '204 No Content'}")
print(f"   LIST  as {PRINCIPAL} : 200 OK                  →  200 OK  (read privileges untouched)")

alice → DELETE /namespaces/cat2_demo  →  HTTP 204
alice → GET    /namespaces           →  HTTP 200
   alice sees namespaces: []

Summary (BEFORE → AFTER):
   DROP  as alice : 403 ForbiddenException  →  204 No Content
   LIST  as alice : 200 OK                  →  200 OK  (read privileges untouched)


## What you just saw

- **Polaris = plain Iceberg REST + a control plane.** The catalog API (`CREATE`/`DROP` namespace, `SHOW NAMESPACES` from Spark) is standard Iceberg REST — what any engine speaks. The management API (`/api/management/v1/…`) is Polaris's addition, and that's the entire subject of this notebook.
- **Principals authenticate with their own OAuth2 credentials.** Real jobs get their own `client_id`/`client_secret`, not root's. Alice's break→fix was observed against **her** credentials.
- **Grants are enum-scoped nouns.** `NAMESPACE_DROP` vs `TABLE_DROP` vs `VIEW_DROP` are separate privileges. Guessing wrong gives you a 403 that literally names the correct enum.
- **The 403 body is the design.** Polaris tells you *why* — no log-diving required.

### Known cut in this lab

- **DROP a namespace, not a table.** MiniStack Community's STS emulator can't vend usable subscoped S3 credentials, and Polaris's write path needs that AssumeRole loop. So creating an *Iceberg table* through Polaris fails at the SDK, and we can't stage a `TABLE_DROP` break. The RBAC semantics are identical — same wire call, same grants-check code path against `TABLE_DROP` — so the lesson holds, and this is called out openly. If you're on LocalStack/MiniStack Pro (which implements STS), swap `NAMESPACE_DROP` → `TABLE_DROP` everywhere above.
- **For governed marts destinations, see [CAT-5](cat5_federation.ipynb).** Nessie → **Glue** is the write-path that lands in this repo today; Polaris stays as the RBAC catalog.

**Next:** the raw HTTP plumbing behind all of this — including the OAuth2 client-credentials flow — lives in [`catalog_api_playground.ipynb`](catalog_api_playground.ipynb).

## Teardown

Delete the principal, principal-role, and catalog-role we created so the catalog is clean for peer modules.

In [9]:
# Idempotent: alice already dropped cat2_demo in Prove; the reset function
# handles principals + roles + demo namespace.
reset()
print("cleaned up alice, reader_role, reader catalog-role, and any leftover cat2_demo.")

cleaned up alice, reader_role, reader catalog-role, and any leftover cat2_demo.
